In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"
import pandas as pd
import numpy as np
import random
import pickle
import itertools
import gc
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
from tqdm import tqdm
from sklearn.metrics import r2_score
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split


from data.data_loader_extend import EPCDataset, UMassDataset
from models.utils_extend import create_model, train_for_long_term_forecast, train_for_short_term_forecast, evaluate_for_long_term_forecast, evaluate_for_short_term_forecast


from explainers.utils_extend import get_explainer, unpack_eval_for_single

In [2]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [3]:
dataset_params = {
    # Long-term forecast
    'long_term_length' : 90*24,           # 입력 길이 (예: 90일치, 1시간 단위)
    'long_term_pred_length' : 256,        # 출력 길이 (예: 256시간)

    # Short-term forecast
    'short_term_length' : 30*24,          # 입력 길이 (예: 30일치)
    'short_term_pred_length' : 7*24,      # 출력 길이 (예: 7일치)

    # 단일 short-term (LSTM/GRU/CNN-LSTM 전용)
    'sequence_length' : 24*30,            # 입력 길이 (예: 30일치)
    'prediction_length' : 24              # 출력 길이 (예: 하루치)
}


is_long_term_forecast = True
# dataset_name = 'epc'
dataset_name = 'umass'

In [4]:
def load_dataset(params,
                 dataset_type="epc",
                 file_path=None,
                 is_long_term_forecast=True,
                 target_name=None,
                 categorical_features=('icon',)):
    """
    params: {
        # long/short 모두 쓰는 경우
        "long_term_length": int,
        "long_term_pred_length": int,
        "short_term_length": int,
        "short_term_pred_length": int,

        # single 모드만 쓰는 경우
        "sequence_length": int,
        "prediction_length": int,
    }

    dataset_type: "epc" | "umass"
    target_name:  타깃 컬럼명 하나만 명시(Optional). None이면
                  EPC→Global_active_power, UMass→use_total, 없으면 첫 숫자형.
    categorical_features: UMass에서 임베딩 인덱스로 뽑을 범주형 컬럼 tuple
    """
    selected_features = {}

    # 어떤 클래스를 쓸지 선택
    if dataset_type.lower() == "epc":
        DatasetClass = EPCDataset
        is_umass = False
        ds_extra_kwargs = {}  # EPC는 범주형 없음
    elif dataset_type.lower() == "umass":
        DatasetClass = UMassDataset  # 주의: UMassDataset가 아니라 UmassDataset
        is_umass = True
        ds_extra_kwargs = {"categorical_features": categorical_features}
    else:
        raise ValueError(f"Unknown dataset_type: {dataset_type}")

    if is_long_term_forecast:
        # ----- Long -----
        ds_long = DatasetClass(
            file_path=file_path,
            sequence_length=params["long_term_length"],
            prediction_length=params["long_term_pred_length"],
            target_name=target_name,
            **ds_extra_kwargs
        )
        long_out = ds_long.load_data()
        target_scaler_long = ds_long.target_scaler

        # ----- Short -----
        ds_short = DatasetClass(
            file_path=file_path,
            sequence_length=params["short_term_length"],
            prediction_length=params["short_term_pred_length"],
            target_name=target_name,
            **ds_extra_kwargs
        )
        short_out = ds_short.load_data()
        target_scaler_short = ds_short.target_scaler

        target_scaler = (target_scaler_long, target_scaler_short)

        if is_umass:
            # UMass: 6개 반환
            Xn_tr_L, Xi_tr_L, y_tr_L, Xn_ev_L, Xi_ev_L, y_ev_L = long_out
            Xn_tr_S, Xi_tr_S, y_tr_S, Xn_ev_S, Xi_ev_S, y_ev_S = short_out

            train_data    = ((Xn_tr_L, Xi_tr_L), (Xn_tr_S, Xi_tr_S))
            train_targets = (y_tr_L, y_tr_S)
            eval_data     = ((Xn_ev_L, Xi_ev_L), (Xn_ev_S, Xi_ev_S))
            eval_targets  = (y_ev_L, y_ev_S)
        else:
            # EPC: 4개 반환
            train_long, train_targets_long, eval_long, eval_targets_long = long_out
            train_short, train_targets_short, eval_short, eval_targets_short = short_out

            train_data    = (train_long,  train_short)
            train_targets = (train_targets_long, train_targets_short)
            eval_data     = (eval_long,   eval_short)
            eval_targets  = (eval_targets_long, eval_targets_short)

        # auto-selected numeric feature names를 우선 제공
        get_names = lambda ds: (ds.get_numeric_feature_names()
                                if hasattr(ds, "get_numeric_feature_names")
                                else getattr(ds, "selected_features", []))
        selected_features["long"]  = get_names(ds_long)
        selected_features["short"] = get_names(ds_short)

    else:
        # ----- Single (short only) -----
        ds_short = DatasetClass(
            file_path=file_path,
            sequence_length=params["sequence_length"],
            prediction_length=params["prediction_length"],
            target_name=target_name,
            **ds_extra_kwargs
        )
        out = ds_short.load_data()
        target_scaler = ds_short.target_scaler

        if is_umass:
            Xn_tr, Xi_tr, y_tr, Xn_ev, Xi_ev, y_ev = out
            train_data    = (Xn_tr, Xi_tr)
            train_targets = y_tr
            eval_data     = (Xn_ev, Xi_ev)
            eval_targets  = y_ev
        else:
            train_seq, train_tgt, eval_seq, eval_tgt = out
            train_data,  train_targets = train_seq, train_tgt
            eval_data,   eval_targets  = eval_seq,  eval_tgt

        if hasattr(ds_short, "get_numeric_feature_names"):
            selected_features["single"] = ds_short.get_numeric_feature_names()
        else:
            selected_features["single"] = getattr(ds_short, "selected_features", [])

    return train_data, train_targets, eval_data, eval_targets, selected_features, target_scaler


In [5]:
train_data, train_targets, eval_data, eval_targets, selected_features, target_scaler = load_dataset(
    params=dataset_params,
    dataset_type=dataset_name,
    file_path="data/umass/HomeA/HomeA_with_weather.csv",
    is_long_term_forecast=is_long_term_forecast,
    target_name="use_total",               # 생략 가능(기본: use_total)
    categorical_features=('icon',)         # UMass에서 icon 임베딩 사용
)


print(selected_features)
# print(len(selected_features['single']))
# print(len(selected_features['long']))
# print(len(selected_features['short']))

{'long': ['use_total', 'use_m2', 'FurnaceHRV_m2', 'CellarOutlets_m2', 'WashingMachine_m2', 'FridgeRange_m2', 'DisposalDishwasher_m2', 'KitchenLights_m2', 'BedroomOutlets_m2', 'BedroomLights_m2', 'MasterOutlets_m2', 'MasterLights_m2', 'DuctHeaterHRV_m2', 'use_m3', 'ElectricRange_m3', 'Dryer_m3', 'GarageMudroomLights_m3', 'DiningRoomOutlets_m3', 'MudroomOutlets_m3', 'MasterBathOutlets_m3', 'GarageOutlets_m3', 'BasementOutdoorOutlets_m3', 'use_m4', 'KitchenDenLights_m4', 'MasterBedBathLights_m4', 'MasterOutlets_m4', 'DenOutdoorLights_m4', 'DenOutlets_m4', 'RearBasementLights_m4', 'KitchenOutletsEast_m4', 'KitchenOutletsSouth_m4', 'DishwasherDisposalSinkLight_m4', 'Refrigerator_m4', 'Microwave_m4', 'OfficeLights_m4', 'temperature', 'humidity', 'visibility', 'apparenttemperature', 'pressure', 'windspeed', 'cloudcover', 'windbearing', 'precipintensity', 'dewpoint', 'precipprobability', 'sin_hour', 'cos_hour', 'sin_day', 'cos_day', 'sin_month', 'cos_month'], 'short': ['use_total', 'use_m2', '

In [6]:
def load_trained_model(
    params,
    is_long_term_forecast=True,
    oversample_eval=True,
    icon_embed_dim=8,
    path_suffix=None
):
    if params is None:
        raise ValueError("params must be provided")

    model_name    = params['model_name']
    hidden_size   = params['hidden_size']
    num_layers    = params['num_layers']
    dropout       = params['dropout']
    num_epochs    = params['num_epochs']
    batch_size    = params['batch_size']
    learning_rate = params['learning_rate']
    patience      = params['patience']
    mse_decay     = params.get('mse_decay', False)

    output_size = {
        'long'  : dataset_params.get('long_term_pred_length', dataset_params.get('prediction_length', 24)),
        'short' : dataset_params.get('short_term_pred_length', dataset_params.get('prediction_length', 24)),
        'single': dataset_params.get('prediction_length', 24),
    }

    mse_alpha, mse_beta = (0.3, 1.0) if mse_decay else (1.0, 1.0)

    icon_vocab_size = None

    if is_long_term_forecast:
        # EPC: train_data = (XL, XS)
        # UMass: train_data = ((XnL, XiL), (XnS, XiS))
        train_long, train_short = train_data
        is_umass = isinstance(train_long, (tuple, list)) and isinstance(train_short, (tuple, list))
    
        if is_umass:
            XnL, XiL = train_long
            XnS, XiS = train_short
            in_long  = XnL.shape[2]
            in_short = XnS.shape[2]
            icon_vocab_size = int(torch.max(torch.stack([XiL.max(), XiS.max()])).item()) + 1
        else:
            XL, XS = train_long, train_short
            in_long  = XL.shape[2]
            in_short = XS.shape[2]
            icon_vocab_size = None  # EPC
    
        long_output_size  = dataset_params['long_term_pred_length']
        short_output_size = dataset_params['short_term_pred_length']
    
        model = create_model(
            model_name=model_name,
            # 장·단기 입력/출력
            long_input_size_num=in_long,
            short_input_size_num=in_short,
            long_output_size=long_output_size,
            short_output_size=short_output_size,
            # 공통
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            # 아이콘(UMass만 유효)
            icon_vocab_size=icon_vocab_size,
            icon_embed_dim=icon_embed_dim,
            # 참고: 길이(모델이 필요로 하면)
            long_term_length=dataset_params['long_term_length'],
            short_term_length=dataset_params['short_term_length']
        )

        suffix = path_suffix or ("umass" if icon_vocab_size else "epc")
        model_path = './trained_models/{}_{}_long{}_short{}_pred{}_hs{}_nl{}_a{}_b{}.pth'.format(
            model_name, suffix,
            dataset_params['long_term_length'], dataset_params['short_term_length'],
            dataset_params['long_term_pred_length'],
            hidden_size, num_layers, mse_alpha, mse_beta
        )
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        print("Model path:", model_path)

        # ---------------------------
        # 로드 or 학습
        # ---------------------------
        if os.path.exists(model_path):
            print(f"Loading the pre-trained {model_name} ...")
            model.load_state_dict(torch.load(model_path, map_location='cpu'))
            model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
        else:
            print(f"{model_name} not found. Training a new one (long-term)...")
            train_for_long_term_forecast(
                model=model,
                model_name=model_name,
                train_data=train_data,          # 전역 (EPC: (XL, XS) / UMass: ((XnL, XiL), (XnS, XiS)))
                train_targets=train_targets,    # (yL, yS)
                eval_data=eval_data,            # 구조 동일
                eval_targets=eval_targets,      # (yL, yS)
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience,
                oversample_eval=oversample_eval,  # 평가시 오버샘플 여부
                alpha=mse_alpha,
                beta=mse_beta
            )

        
    else:
        # EPC: train_data = Xn
        # UMass: train_data = (Xn, Xi)
        is_umass = isinstance(train_data, (tuple, list))
    
        if is_umass:
            Xn, Xi = train_data
            in_single = Xn.shape[2]
            icon_vocab_size = int(Xi.max().item()) + 1
        else:
            Xn = train_data
            in_single = Xn.shape[2]
            icon_vocab_size = None
    
        output_size_single = dataset_params.get('prediction_length', 24)
    
        model = create_model(
            model_name=model_name,
            input_size_num=in_single,
            output_size=output_size_single,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            icon_vocab_size=icon_vocab_size,
            icon_embed_dim=icon_embed_dim
        )

        suffix = path_suffix or ("umass" if icon_vocab_size else "epc")
        model_path = './trained_models/{}_{}_seq{}_pred{}_hs{}_nl{}.pth'.format(
            model_name, suffix,
            dataset_params['sequence_length'], dataset_params['prediction_length'],
            hidden_size, num_layers
        )
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        print("Model path:", model_path)
    
        if os.path.exists(model_path):
            print(f"Loading the pre-trained {model_name} ...")
            model.load_state_dict(torch.load(model_path, map_location='cpu'))
            model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
        else:
            print(f"{model_name} not found. Training a new one...")
            train_for_short_term_forecast(
                model=model,
                model_name=model_name,
                train_sequences=train_data,
                train_targets=train_targets,
                eval_sequences=eval_data,
                eval_targets=eval_targets,
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience
            )

    return model, model_path

In [7]:
# hz_list = [128, 192]
# nl_list = [3, 4]
# save_path = 'results/transformer_umass_performance.txt'

# for hz, nl in itertools.product(hz_list, nl_list):
#     model_params = {
#         'model_name'   : 'LS_Transformer',
#         'hidden_size'  : hz,    # = d_model
#         'num_layers'   : nl,      # = encoder layers
#         'dropout'      : 0.15,
#         'num_epochs'   : 200,
#         'batch_size'   : 64,
#         'learning_rate': 0.0001, # AdamW/Adam 모두 무난
#         'patience'     : 15,
#         'mse_decay'    : False    # 기존 스케줄 쓰는 경우 호환
#     }

#     model, model_path = load_trained_model(
#         params=model_params,
#         is_long_term_forecast=is_long_term_forecast,
#         oversample_eval=True  # 평가셋 오버샘플 여부(기존 정책)
#     )

#     print("Evaluating the model...")
#     with torch.no_grad():
#         if is_long_term_forecast:
#             # -------- Long-term forecast 평가 --------
#             results = evaluate_for_long_term_forecast(
#                 model=model,
#                 eval_data=eval_data,
#                 eval_targets=eval_targets,
#                 model_name=model_params['model_name'],
#                 batch_size=model_params['batch_size'],
#                 oversample_eval=False,     # 기존 정책 유지
#                 # target_scaler=target_scaler
#             )
#         else:
#             # -------- Short-term forecast 평가 --------
#             # EPC: eval_data = X, eval_targets = y
#             # UMass: eval_data = (Xn, Xi), eval_targets = y
#             results = evaluate_for_short_term_forecast(
#                 model=model,
#                 eval_sequences=eval_data,     # EPC는 X, UMass는 (Xn, Xi)
#                 eval_targets=eval_targets,
#                 model_name=model_params['model_name'],
#                 batch_size=model_params['batch_size'],
#                 # target_scaler=target_scaler
#             )
        
#     print("Evaluation results:", results)

#     with open(save_path, "a") as f:  
#         print_res = '{}\n{}\n\n'.format(model_path, results)
#         f.write(print_res)

#     del model
#     torch.cuda.empty_cache()
#     gc.collect()

In [8]:
hz_list = [128, 192]
nl_list = [3, 4]

In [9]:
model_params = {
    'model_name'  : 'LS_Transformer',
    'hidden_size' : 192,
    'num_layers'  : 4,
    'dropout'     : 0.15,
    'num_epochs'  : 200,
    'batch_size'  : 64,
    'learning_rate': 0.0001,
    'patience'    : 15,
    'mse_decay'   : False
}

model, model_path = load_trained_model(
    params=model_params,
    is_long_term_forecast=is_long_term_forecast,
    oversample_eval=True  # 평가셋 오버샘플 여부(기존 정책)
)

Model path: ./trained_models/LS_Transformer_umass_long2160_short720_pred256_hs192_nl4_a1.0_b1.0.pth
Loading the pre-trained LS_Transformer ...


In [10]:
print("Evaluating the model...")

if is_long_term_forecast:
    # -------- Long-term forecast 평가 --------
    results = evaluate_for_long_term_forecast(
        model=model,
        eval_data=eval_data,
        eval_targets=eval_targets,
        model_name=model_params['model_name'],
        batch_size=model_params['batch_size'],
        oversample_eval=True,     # 기존 정책 유지
        # target_scaler=target_scaler
    )
else:
    # -------- Short-term forecast 평가 --------
    # EPC: eval_data = X, eval_targets = y
    # UMass: eval_data = (Xn, Xi), eval_targets = y
    results = evaluate_for_short_term_forecast(
        model=model,
        eval_sequences=eval_data,     # EPC는 X, UMass는 (Xn, Xi)
        eval_targets=eval_targets,
        model_name=model_params['model_name'],
        batch_size=model_params['batch_size'],
        # target_scaler=target_scaler
    )

print("Evaluation results:", results)

Evaluating the model...
[Short] R²: 0.8263, Adj.R²: 0.8245, SMAPE: 23.39, MASE: 0.8395
[Long ] R²: 0.8374, Adj.R²: 0.8357, SMAPE: 23.67, MASE: 0.7896
Evaluation results: {'R2 Short': 0.8263331651687622, 'Adjusted R2 Short': 0.8245378022639462, 'SMAPE Short': np.float32(23.393938), 'MASE Short': np.float32(0.83951706), 'R2 Long': 0.8373554944992065, 'Adjusted R2 Long': 0.8356740801282242, 'SMAPE Long': np.float32(23.674055), 'MASE Long': np.float32(0.7896458)}


## Feature 추출
### LIME

In [11]:
important_features_dict = {}

if is_long_term_forecast:
    num_samples=eval_data[0][0].shape[0]
    num_of_features = len(selected_features['long'])
else:
    num_samples=len(eval_data[0])
    num_of_features = len(selected_features['single'])

if is_long_term_forecast:
    sequence_length = {
        'long': dataset_params['long_term_length'],
        'short': dataset_params['short_term_length']
    }
    input_size = {
        # 'long': len(features_for_6h)+6,
        'long': num_of_features,
        'short': num_of_features,
        'single': num_of_features
    }
    output_size = {
        'long': dataset_params['long_term_pred_length'],
        'short': dataset_params['short_term_pred_length'],
        'single': dataset_params['prediction_length']
    }
else:
    sequence_length = {
        'single': dataset_params['sequence_length']
    }
    input_size = {
        'single': num_of_features
    }
    output_size = {
        'single': dataset_params['prediction_length']
    }


num_samples=1000  # perturbation 개수
num_datapoints = 500 # 몇개 평가 샘플 볼지
top_n = int(num_of_features)


In [12]:
explainer_type = 'LIME' 

explainer = get_explainer(
    explainer_type=explainer_type, 
    model=model, 
    device=device, 
    train_data=train_data,  
    sequence_length=sequence_length,
    input_size=input_size,
    selected_features=selected_features,
    is_long_term_forecast=is_long_term_forecast,
    scaler=MinMaxScaler()
)

In [13]:
if is_long_term_forecast:
    important = explainer.extract_important_features(
        eval_long=eval_data[0],      # 또는 (X_long, ICON_long) numpy
        eval_short=eval_data[1],     # 또는 (X_short, ICON_short) numpy
        # 샘플링 전략 (택1)
        num_datapoints=num_datapoints,          # 고정 개수 (권장 256~512)
        # sample_fraction=0.10,      # 10% 쓰고 싶으면 이걸로
        random_state=42,
        top_n_for_long=top_n,
        top_n_for_short=top_n,
        early_stop=False, check_interval=64, stability_k=20, rho_thresh=0.95
    )
    
else:
    eval_X, eval_ICON, eval_len = unpack_eval_for_single(eval_data)  # ★ 수정

    important = explainer.extract_important_features(
        eval_data=(eval_X, eval_ICON),
        num_datapoints=num_datapoints,
        top_n=top_n,
        random_state=42
    )

Processing samples: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [7:31:37<00:00, 54.19s/it]


In [1]:
if is_long_term_forecast:
    def save_important_features_longterm(important_features, output_file, model_path):
        with open(output_file, 'a') as file:
            file.write("\n\n{}\n".format(model_path))

            file.write("Important Long-term Features:\n")
            for feature_name, score in important_features["long"]:
                file.write(f"{feature_name}: {score}\n")

            file.write("\nImportant Short-term Features:\n")
            for feature_name, score in important_features["short"]:
                file.write(f"{feature_name}: {score}\n")

    output_file = "results/umass_trans_important_features_for_lf.txt"
    save_important_features_longterm(important, output_file, model_path)

    print(f"Important long/short-term features saved to {output_file}")
    
else:
    def save_important_features(important_features, output_file, model_path):
        with open(output_file, 'a') as file:
            file.write("\n\n{}\n".format(model_path))
            file.write("Important Features:\n")
            for feature_name, score in important_features["single"]:
                file.write(f"{feature_name}: {score}\n")
    
    output_file = "results/umass_trans_important_features_for_sf.txt"
    save_important_features(important, output_file, model_path)
    
    print(f"Important features saved to {output_file}")

NameError: name 'is_long_term_forecast' is not defined